In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import importlib
import utils.alr_plotting as alr_plotting

from bluemath_tk.teslakit.alr import ALR_WRP
from bluemath_tk.core.io import load_model
from bluemath_tk.teslakit.mjo import MJO_Categories, MJO_Phases


In [ ]:
runModel = False

#### Load Data (PCs MSLP - ERA5 & Wave Clusters)

In [ ]:
# Daily PCs
waveClusters_and_pcs = pd.read_csv('outputs/PCA/pcs_clusters_ordered_ERA5.csv', index_col=0, parse_dates=False)
waveClusters_and_pcs.index = pd.to_datetime(waveClusters_and_pcs.index).floor('D')

# Align PCs and BMUs to start on 1980-01-01
start_date = pd.Timestamp('1980-01-01')
waveClusters_and_pcs = waveClusters_and_pcs.loc[start_date:]

# PCs used as covariates (all columns except BMUs)
pcs_daily = waveClusters_and_pcs.iloc[:, :-1]

#### Prepare_data

In [ ]:
pcs_daily = (
    pcs_daily
    .sort_index()
    .interpolate(method="time")    # or .ffill().bfill(), or a domain-specific fill
)

In [ ]:
pcs_daily.head()

In [ ]:
waveClusters_and_pcs.head()

In [ ]:
covar_ds = pcs_daily.to_xarray()
covar_ds = covar_ds.rename({'index':'time'})
covar_ds

#### Covariates

In [ ]:
nPCs = pcs_daily.shape[1]
cov_names_list = pcs_daily.columns.tolist()
cov_list = []

for i in range(nPCs):
    cov_list.append(covar_ds[cov_names_list[i]].values)
cov_combined = np.hstack([cov_list]).T  # shape (n_covariates, n_times)

cov_names_list = pcs_daily.columns.tolist()
new_cov_norm = xr.DataArray(
    cov_combined,
    dims=["time", "n_covariates"],
    coords={
        "time": covar_ds.time.values,
        "n_covariates": np.arange(cov_combined.shape[1])  # [0, 1, 2, 3, 4]
    },
    name="cov_norm"
)

covariates_ds = new_cov_norm.to_dataset()
covariates_ds = (covariates_ds - covariates_ds.mean(dim="time")) / covariates_ds.std(
    dim="time"
)
covariates_ds["cov_names"] = (("n_covariates"), cov_names_list)

In [ ]:
covariates_ds

#### ALR

In [ ]:
alr_model = ALR_WRP(
    p_base="outputs/Emulator/ALR"
)

alr_terms = {
    "mk_order": 0,
    "constant": True,
    "long_term": False,
    # "seasonality": (True, [2,4,6]),
    "covariates": (
        True,
        covariates_ds,
    ),
}

nTT = waveClusters_and_pcs['kma_bmus'].nunique() 
df_bmus_fit = pd.DataFrame()
df_bmus_fit['time'] = covar_ds.time.values
df_bmus_fit['bmus'] = waveClusters_and_pcs['kma_bmus'].values.astype(int)+1
xds_bmus_fit = df_bmus_fit.set_index('time').to_xarray()

alr_model.SetFitData(
    cluster_size=int(nTT),
    xds_bmus_fit=xds_bmus_fit,
    d_terms_settings=alr_terms,
)

In [ ]:
import xarray as xr
from pathlib import Path
from bluemath_tk.teslakit.alr import ALR_WRP

alr_model = ALR_WRP("outputs/Emulator/ALR")
alr_model.LoadModel()

# Skip corrupted/missing xds_input.nc (takes hours to rebuild)
p_in = Path("outputs/Emulator/ALR/xds_input.nc")
if p_in.exists() and p_in.stat().st_size > 0:
    alr_model.xds_bmus_fit = xr.open_dataset(p_in)
    alr_model.cluster_size = int(alr_model.xds_bmus_fit.attrs.get("cluster_size", nTT))
else:
    alr_model.xds_bmus_fit = None
    alr_model.cluster_size = int(nTT)  # or 80, if fixed in your setup
    print("xds_input.nc missing/empty -> continuing without observed BMUs.")

In [ ]:
import seaborn as sns

pvals = alr_model.model.pvalues

plt.figure(figsize=(14, 10))
ax = sns.heatmap(pvals,cmap="coolwarm_r",linewidths=0.5,linecolor='gray',cbar_kws={'label': 'p-value'})

# Add black dots for significant coefficients
for i in range(pvals.shape[0]):
    for j in range(pvals.shape[1]):
        if pvals.iloc[i, j] < 0.05:
            plt.scatter(j + 0.5, i + 0.5, color='black', s=12.5, marker='o')  # s is size

nTT = int(waveClusters_and_pcs['kma_bmus'].nunique())  # 81
cluster_labels = np.arange(0, nTT-1)  # clusters 1..80, baseline = 81
ax.set_xticks(np.arange(pvals.shape[1]) + 0.5)
ax.set_xticklabels(cluster_labels, rotation=90)
plt.title("ALR Model Coefficients p-values (dots = p < 0.05)", fontsize=16)
plt.ylabel("Dependent Variable")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()

#### Emulate Historical Hindcast (1980 - 2023)

In [ ]:
pcs_per_sim_list = pd.read_csv(
    '../03A_Stochastic_GCMs_NC/outputs/cmip6_models/ec_earth3_veg_lr/ssp585/mslp_ec_earth3_veg_lr_ssp585_pcs_95.csv',
    index_col=0,
    parse_dates=True
)
pcs_per_sim_list.index = pd.to_datetime(pcs_per_sim_list.index).floor('D')

In [ ]:
# Support both: list of DataFrames or single DataFrame
if isinstance(pcs_per_sim_list, pd.DataFrame):
    first_df = pcs_per_sim_list
    pcs_per_sim_list = [pcs_per_sim_list]
else:
    first_df = pcs_per_sim_list[0]

time = first_df.index
cov_names = first_df.columns
n_cov = len(cov_names)
n_sims = len(pcs_per_sim_list)

cov_stack = np.stack([df.values for df in pcs_per_sim_list])

covar_sim = xr.Dataset(
    data_vars=dict(
        cov_values=(("n_sim", "time", "n_covariates"), cov_stack),
        cov_names=(("n_sim", "n_covariates"),
                   np.tile(cov_names.values, (n_sims, 1)))
    ),
    coords=dict(
        n_sim=np.arange(n_sims),
        time=time,
        n_covariates=np.arange(n_cov),
    )
)

In [ ]:
covar_sim

In [ ]:
print("covar_sim dims:", covar_sim.dims)
print("covar_sim.sizes:", covar_sim.sizes)
ds_i = covar_sim.isel(n_sim=0)
print("ds_i.cov_values.shape:", ds_i.cov_values.shape)   # (T_cov, n_cov)
print("time_sim length:", len(covar_sim.time.values))

In [ ]:
n_sims = covar_sim.sizes['n_sim']  # or len(covar_sim.n_sim)

simulated_daily_bmus_list = []
for i in range(n_sims):
    simulated_daily_bmus = alr_model.Simulate(
        num_sims=1,
        time_sim=covar_sim.time.values,
        xds_covars_sim=covar_sim.isel(n_sim=i),
    )
    simulated_daily_bmus_list.append(simulated_daily_bmus)

simulated_daily_bmus = xr.concat(simulated_daily_bmus_list, dim="n_sim")

In [ ]:
sim = simulated_daily_bmus
print(sim.nan_days_per_sim.values)
print(sim.nan_pct_per_sim.values)
print(sim.attrs["nan_days_total"], sim.attrs["nan_days_total_pct"])

In [ ]:
# Save figures in project folder (works even if Jupyter was started elsewhere)
save_folder = os.path.join("outputs", "Figures", "ALR/earth3_veg_ssp585")
os.makedirs(save_folder, exist_ok=True)
print("Saving figures to:", save_folder)

l_figs = alr_model.Report_Sim()

# Save all figures
fig_names = ["PerpetualYear.png", "Transitions.png", "Persistences.png"]
saved = []
for fig, name in zip(l_figs, fig_names):
    path = os.path.join(save_folder, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    saved.append(path)
print("Saved", len(saved), "figures:")
for p in saved:
    print(" ", p)

l_figs  # keep figures in notebook output

In [ ]:
# simulated_daily_bmus.to_netcdf('outputs/mnlogit_waveClusters/waveClusters_sim_kmedoids.nc')
# simulated_daily_bmus = xr.open_dataset('outputs/mnlogit_waveClusters/waveClusters_sim_kmedoids.nc')
# simulated_daily_bmus

In [ ]:
# Flatten simulated BMUs for comparison
sim_vals = simulated_daily_bmus.evbmus_sims.values.flatten()-1
fit_vals = xds_bmus_fit.bmus.values-1

bmus = np.arange(0,nTT,1)

# Compute densities (or frequencies)
sim_counts = [np.mean(sim_vals == bmu) for bmu in bmus]
fit_counts = [np.mean(fit_vals == bmu) for bmu in bmus]

# Width for bars
width = 0.35
x = np.arange(len(bmus))

plt.figure(figsize=(18,5))
plt.bar(x - width/2, sim_counts, width, label='Simulated', color='coral', edgecolor='black')
plt.bar(x + width/2, fit_counts, width, label='Observed', color='cornflowerblue', edgecolor='black')

# label only every 5th BMU and rotate labels
step = 1
plt.xticks(x[::step], bmus[::step], rotation=90)

plt.xlim(-0.5, len(bmus) - 0.5)
plt.xlabel('BMUS (0–79)')
plt.ylabel('Density')
plt.title('Observed vs Simulated BMUS Distributions')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
# Save figure
import os
save_folder = os.path.join("outputs", "Figures", "ALR/earth3_veg_ssp585")
os.makedirs(save_folder, exist_ok=True)
plt.gcf().savefig(os.path.join(save_folder, "BMUS_Observed_vs_Simulated.png"), dpi=150, bbox_inches="tight")
print("Saved: BMUS_Observed_vs_Simulated.png")
plt.show()

#### Time-series Emulated - Using Random Cluster data from Binwaves Simulation

In [ ]:
import importlib
import os
import utils.bmu_bootstrap_timeseries as bmu_bt
importlib.reload(bmu_bt)
from utils.bmu_bootstrap_timeseries import create_bmu_bootstrap_waves_winds_for_merged_sites

# ---------------------------------------------------------------------------
# Hindcast product: choose one
#   "merged_grids" → full BinWaves merge (includes offshore / fewPoints coords)
#   "500m"         → coastal 500 m points only
# ---------------------------------------------------------------------------
product = "merged_grids"  # "merged_grids" | "500m"

PRODUCTS = {
    "merged_grids": {
        "source_dir": "../02_Wind_Metamodel/outputs/NorthCarolina",
        "output_dir": "outputs/earth3_veg_ssp585/OffshorePoints",
    },
    # "500m": {
    #     "source_dir": "outputs/merged_500m_binwaves_bmus",
    #     # or: "../02_Wind_Metamodel/outputs/merged_500m_binwaves_bmus"
    #     "output_dir": "outputs/earth3_veg_ssp585/OffshorePoints",
    # },
}
if product not in PRODUCTS:
    raise ValueError(f"Unknown product={product!r}. Choose one of {list(PRODUCTS)}")

source_dir = PRODUCTS[product]["source_dir"]
output_dir = PRODUCTS[product]["output_dir"]
print(f"product={product}\nsource_dir={source_dir}\noutput_dir={output_dir}")

available_variables = sorted({
    os.path.splitext(fn)[0].split("_", 1)[0]
    for fn in os.listdir(source_dir)
    if fn.endswith(".nc")
})
selected_variables = available_variables

# Site selection (pick one):
# - None  → all sites in source_dir files
# - one point: selected_coordinate=(36.1942, -75.7316)
# - several points: selected_coordinate=[(lat, lon), ...]  (nearest site each; duplicates dropped)
# - or by index: selected_site_indices=[12, 45, 90]
selected_coordinate = [
    (36.400, -75.200),
    (36.300, -75.400),
    (36.258, -75.593),
    (36.187500, -74.875000),
    (35.625000, -74.875000),
    (35.375000, -75.187500),
    (34.812500, -75.812500),
    (34.125000, -77.000000),
    (33.375000, -78.375000),
]

selection = create_bmu_bootstrap_waves_winds_for_merged_sites(
    hindcast_waves_winds_dir=source_dir,
    simulated_daily_bmus=simulated_daily_bmus,
    waveclusters_and_pcs=waveClusters_and_pcs,
    output_dir=output_dir,
    monthly_conditioning=True,
    skip_existing_outputs=True,
    sim_indices=0,
    n_workers=6,  # parallel files (~5 GB RAM per worker)
    selected_coordinate=selected_coordinate,
)


In [ ]:
import importlib
importlib.reload(alr_plotting)
daily_pairs, save_folder = alr_plotting.plot_from_folder(
    data_folder="outputs/earth3_veg_ssp585/OffshorePoints",
    point_lat=36.1942,
    point_lon=-75.7316,
    variables=["hs", "tp", "dp"],
    historical_folder="../02_Wind_Metamodel/outputs/NorthCarolina",
    shoreline_orientation_deg=72,
    original_band=None,
    plot_non_overlap_timeseries=True,
)


In [ ]:
import importlib
importlib.reload(alr_plotting)
daily_pairs, save_folder = alr_plotting.plot_from_folder(
    data_folder="outputs/earth3_veg_ssp585/OffshorePoints",
    point_lat=33.8883,
    point_lon=-78.1551,
    variables=["hs", "tp", "dp"],
    historical_folder="../02_Wind_Metamodel/outputs/NorthCarolina",
    shoreline_orientation_deg=188,
    original_band=None,
    plot_non_overlap_timeseries=True,
)


In [ ]:
import importlib
importlib.reload(alr_plotting)
daily_pairs, save_folder = alr_plotting.plot_from_folder(
    data_folder="outputs/earth3_veg_ssp585/OffshorePoints",
    point_lat=36.1875,
    point_lon=-74.875,
    variables=["hs", "tp", "dp"],
    historical_folder="../02_Wind_Metamodel/outputs/NorthCarolina",
    shoreline_orientation_deg=72,
    original_band=None,
    plot_non_overlap_timeseries=True,
)


In [ ]:
import importlib
importlib.reload(alr_plotting)
daily_pairs, save_folder = alr_plotting.plot_from_folder(
    data_folder="outputs/earth3_veg_ssp585/OffshorePoints",
    point_lat=36.4,
    point_lon=-75.2,
    variables=["hs", "tp", "dp"],
    historical_folder="../02_Wind_Metamodel/outputs/NorthCarolina",
    shoreline_orientation_deg=72,
    original_band=None,
    plot_non_overlap_timeseries=True,
)


In [ ]:
import importlib
importlib.reload(alr_plotting)
daily_pairs, save_folder = alr_plotting.plot_from_folder(
    data_folder="outputs/earth3_veg_ssp585/OffshorePoints",
    point_lat=36.3,
    point_lon=-75.4,
    variables=["hs", "tp", "dp"],
    historical_folder="../02_Wind_Metamodel/outputs/NorthCarolina",
    shoreline_orientation_deg=72,
    original_band=None,
    plot_non_overlap_timeseries=True,
)


In [ ]:
import importlib
importlib.reload(alr_plotting)
daily_pairs, save_folder = alr_plotting.plot_from_folder(
    data_folder="outputs/earth3_veg_ssp585/OffshorePoints",
    point_lat=36.258,
    point_lon=-75.593,
    variables=["hs", "tp", "dp"],
    historical_folder="../02_Wind_Metamodel/outputs/NorthCarolina",
    shoreline_orientation_deg=72,
    original_band=None,
    plot_non_overlap_timeseries=True,
)


In [ ]:
import importlib
importlib.reload(alr_plotting)
daily_pairs, save_folder = alr_plotting.plot_from_folder(
    data_folder="outputs/earth3_veg_ssp585/OffshorePoints",
    point_lat=33.375,
    point_lon=-78.375,
    variables=["hs", "tp", "dp"],
    historical_folder="../02_Wind_Metamodel/outputs/NorthCarolina",
    shoreline_orientation_deg=170,
    original_band=None,
    plot_non_overlap_timeseries=True,
)
